# Шаг 1. Зафиксировать этот результат как retrieval diagnostic

In [3]:
import json
import math
import re
import argparse
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
from dateutil import parser as dtparser


# ============================================================
# Optional sentence-transformers support
# ============================================================

HAS_SENTENCE_TRANSFORMERS = False

try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except Exception:
    HAS_SENTENCE_TRANSFORMERS = False


# ============================================================
# Data structures
# ============================================================

@dataclass
class TemporalEvent:
    turn: int
    speaker: str
    observation_time: Optional[str]
    event_time: Optional[str]
    valid_from: Optional[str]
    valid_to: Optional[str]
    status: str
    text: str


@dataclass
class TimeBoundExample:
    id: str
    task_type: str
    difficulty: str
    current_time: str
    history: List[TemporalEvent]
    query: str
    gold_answer: str
    gold_evidence_turns: List[int]
    required_temporal_operation: str
    temporal_trap: str
    explanation: str


@dataclass
class MemoryItem:
    example_id: str
    turn: int
    speaker: str
    text: str
    observation_time: Optional[Any]
    event_time: Optional[Any]
    valid_from: Optional[Any]
    valid_to: Optional[Any]
    status: str
    raw: Dict[str, Any]


@dataclass
class RetrievalResult:
    turn: int
    text: str
    score: float
    semantic_score: float
    temporal_score: float
    status_penalty: float
    validity_label: str
    status: str


@dataclass
class Prediction:
    id: str
    task_type: str
    difficulty: str
    query: str
    gold_answer: str
    predicted_answer: str
    gold_evidence_turns: List[int]
    predicted_evidence_turns: List[int]
    answer_correct: bool
    evidence_precision: float
    evidence_recall: float
    evidence_f1: float
    contradiction: bool
    retrieved: List[Dict[str, Any]]


# ============================================================
# JSONL loading
# ============================================================

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except Exception as e:
                print(f"Bad JSON line {line_no}: {e}")

    return rows


def parse_example(obj: Dict[str, Any]) -> TimeBoundExample:
    history = []

    for ev in obj.get("history", []):
        history.append(
            TemporalEvent(
                turn=int(ev["turn"]),
                speaker=str(ev.get("speaker", "")),
                observation_time=ev.get("observation_time"),
                event_time=ev.get("event_time"),
                valid_from=ev.get("valid_from"),
                valid_to=ev.get("valid_to"),
                status=str(ev.get("status", "")),
                text=str(ev.get("text", "")),
            )
        )

    return TimeBoundExample(
        id=str(obj["id"]),
        task_type=str(obj.get("task_type", "")),
        difficulty=str(obj.get("difficulty", "")),
        current_time=str(obj.get("current_time", "")),
        history=history,
        query=str(obj.get("query", "")),
        gold_answer=str(obj.get("gold_answer", "")),
        gold_evidence_turns=[int(x) for x in obj.get("gold_evidence_turns", [])],
        required_temporal_operation=str(obj.get("required_temporal_operation", "")),
        temporal_trap=str(obj.get("temporal_trap", "")),
        explanation=str(obj.get("explanation", "")),
    )


def load_examples(path: Path, limit: Optional[int] = None) -> List[TimeBoundExample]:
    rows = read_jsonl(path)

    if limit is not None:
        rows = rows[:limit]

    examples = []

    for obj in rows:
        try:
            examples.append(parse_example(obj))
        except Exception as e:
            print(f"Skip malformed example: {e}")

    return examples


# ============================================================
# Datetime utilities
# ============================================================

def parse_time(x: Optional[str]) -> Optional[Any]:
    if x is None:
        return None

    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None

    try:
        return dtparser.parse(str(x))
    except Exception:
        return None


def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .,:;!?")
    return s


def soft_numeric_match(a: str, b: str) -> bool:
    a_norm = normalize_text(a)
    b_norm = normalize_text(b)

    if a_norm == b_norm:
        return True

    nums_a = re.findall(r"\d+", a_norm)
    nums_b = re.findall(r"\d+", b_norm)

    if nums_a and nums_b and nums_a == nums_b:
        return True

    return False


# ============================================================
# Lightweight text encoder without sklearn
# ============================================================

def tokenize(text: str) -> List[str]:
    text = str(text).lower()
    return re.findall(r"[a-zа-я0-9_]+", text, flags=re.I)


def cosine_np(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Computes cosine similarity between one vector a and matrix b.
    a: shape (1, d)
    b: shape (n, d)
    returns: shape (n,)
    """
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    if a.ndim == 1:
        a = a.reshape(1, -1)

    if b.ndim == 1:
        b = b.reshape(1, -1)

    a_norm = np.linalg.norm(a, axis=1, keepdims=True) + 1e-9
    b_norm = np.linalg.norm(b, axis=1, keepdims=True) + 1e-9

    a = a / a_norm
    b = b / b_norm

    return np.dot(b, a[0])


class BaseTextEncoder:
    def fit(self, texts: List[str]) -> None:
        raise NotImplementedError

    def encode(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SimpleTfidfTextEncoder(BaseTextEncoder):
    """
    Minimal TF-IDF encoder without sklearn.
    This avoids pyarrow/sklearn DLL issues on Windows.
    """

    def __init__(self, max_features: int = 50000):
        self.max_features = max_features
        self.vocab: Dict[str, int] = {}
        self.idf: Optional[np.ndarray] = None
        self.is_fitted = False

    def fit(self, texts: List[str]) -> None:
        df = Counter()
        n_docs = len(texts)

        for text in texts:
            tokens = set(tokenize(text))
            for tok in tokens:
                df[tok] += 1

        terms = [tok for tok, _ in df.most_common(self.max_features)]
        self.vocab = {tok: i for i, tok in enumerate(terms)}

        idf = np.zeros(len(self.vocab), dtype=np.float32)

        for tok, idx in self.vocab.items():
            idf[idx] = math.log((1 + n_docs) / (1 + df[tok])) + 1.0

        self.idf = idf
        self.is_fitted = True

    def encode(self, texts: List[str]) -> np.ndarray:
        if not self.is_fitted:
            self.fit(texts)

        assert self.idf is not None

        mat = np.zeros((len(texts), len(self.vocab)), dtype=np.float32)

        for row_idx, text in enumerate(texts):
            toks = tokenize(text)
            if not toks:
                continue

            counts = Counter(toks)
            total = sum(counts.values())

            for tok, cnt in counts.items():
                if tok not in self.vocab:
                    continue

                col = self.vocab[tok]
                tf = cnt / max(total, 1)
                mat[row_idx, col] = tf * self.idf[col]

        return mat


class SentenceTransformerTextEncoder(BaseTextEncoder):
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        if not HAS_SENTENCE_TRANSFORMERS:
            raise RuntimeError(
                "sentence-transformers is not installed. "
                "Install it or use --encoder simple-tfidf."
            )

        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def fit(self, texts: List[str]) -> None:
        return None

    def encode(self, texts: List[str]) -> np.ndarray:
        return np.asarray(
            self.model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )


def build_encoder(
    examples: List[TimeBoundExample],
    encoder_type: str = "simple-tfidf",
    sentence_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
) -> BaseTextEncoder:
    all_texts = []

    for ex in examples:
        all_texts.append(ex.query)
        for ev in ex.history:
            all_texts.append(ev.text)

    if encoder_type == "sentence-transformers":
        encoder = SentenceTransformerTextEncoder(sentence_model_name)
        encoder.fit(all_texts)
        print(f"Using sentence-transformers encoder: {sentence_model_name}")
        return encoder

    if encoder_type == "auto":
        if HAS_SENTENCE_TRANSFORMERS:
            encoder = SentenceTransformerTextEncoder(sentence_model_name)
            encoder.fit(all_texts)
            print(f"Using sentence-transformers encoder: {sentence_model_name}")
            return encoder

        encoder = SimpleTfidfTextEncoder()
        encoder.fit(all_texts)
        print("Using SimpleTfidfTextEncoder without sklearn.")
        return encoder

    if encoder_type == "simple-tfidf":
        encoder = SimpleTfidfTextEncoder()
        encoder.fit(all_texts)
        print("Using SimpleTfidfTextEncoder without sklearn.")
        return encoder

    raise ValueError(f"Unknown encoder type: {encoder_type}")


# ============================================================
# Temporal memory
# ============================================================

class TemporalMemory:
    def __init__(self, example: TimeBoundExample):
        self.example = example
        self.items: List[MemoryItem] = []

        for ev in example.history:
            self.items.append(
                MemoryItem(
                    example_id=example.id,
                    turn=ev.turn,
                    speaker=ev.speaker,
                    text=ev.text,
                    observation_time=parse_time(ev.observation_time),
                    event_time=parse_time(ev.event_time),
                    valid_from=parse_time(ev.valid_from),
                    valid_to=parse_time(ev.valid_to),
                    status=ev.status,
                    raw=asdict(ev),
                )
            )

    def __len__(self) -> int:
        return len(self.items)


# ============================================================
# Temporal scoring
# ============================================================

ACTIVE_STATUSES = {
    "active",
    "scheduled",
    "delayed",
}

NEGATIVE_STATUSES = {
    "expired",
    "cancelled",
    "superseded",
}

HISTORICAL_STATUSES = {
    "historical",
}


def infer_query_mode(query: str, task_type: str) -> str:
    q = normalize_text(query)
    tt = normalize_text(task_type)

    if tt == "elapsed_time_reasoning":
        return "duration"

    if tt == "delayed_observation":
        return "delayed"

    if tt == "periodic_event":
        return "recurrence"

    if "was" in q or "at " in q or "previous" in q or "before" in q:
        return "retrospective"

    if "when" in q or "scheduled" in q or "will" in q or "future" in q:
        return "prospective"

    return "current"


def validity_label(item: MemoryItem, query_time: Any) -> str:
    if query_time is None:
        return "unknown"

    vf = item.valid_from
    vt = item.valid_to

    if vf is not None and query_time < vf:
        return "future"

    if vt is not None and query_time > vt:
        return "expired"

    if vf is not None and vt is not None and vf <= query_time <= vt:
        return "valid"

    if vf is not None and vt is None and query_time >= vf:
        return "valid"

    if vf is None and vt is not None and query_time <= vt:
        return "valid"

    return "unknown"


def temporal_relevance_score(
    item: MemoryItem,
    query_time: Any,
    query_mode: str,
) -> float:
    label = validity_label(item, query_time)

    if query_mode == "current":
        if label == "valid":
            return 1.0
        if label == "future":
            return 0.4
        if label == "expired":
            return 0.1
        return 0.3

    if query_mode == "prospective":
        if item.event_time is not None and query_time is not None:
            if item.event_time >= query_time:
                return 1.0
            return 0.3

        if label in {"valid", "future"}:
            return 0.8

        return 0.2

    if query_mode == "retrospective":
        if label == "valid":
            return 1.0

        if item.status in HISTORICAL_STATUSES:
            return 0.8

        if label == "expired":
            return 0.6

        return 0.3

    if query_mode == "duration":
        if item.event_time is not None:
            return 1.0
        return 0.4

    if query_mode == "delayed":
        if item.event_time is not None and item.observation_time is not None:
            if item.event_time != item.observation_time:
                return 1.0
        return 0.5

    if query_mode == "recurrence":
        if item.valid_from is not None and item.valid_to is not None:
            return 1.0
        return 0.4

    return 0.5


def status_penalty(item: MemoryItem, query_mode: str) -> float:
    status = normalize_text(item.status)

    if status in ACTIVE_STATUSES:
        return 0.0

    if status == "historical":
        if query_mode == "retrospective":
            return 0.0
        return 0.15

    if status == "expired":
        if query_mode == "retrospective":
            return 0.05
        return 0.35

    if status == "superseded":
        if query_mode == "retrospective":
            return 0.10
        return 0.45

    if status == "cancelled":
        return 0.15

    return 0.1


# ============================================================
# Time-aware retriever
# ============================================================

class TimeAwareRetriever:
    """
    Retrieval score:

    S(q, m, t) =
        alpha * semantic_similarity
      + beta  * temporal_relevance
      - gamma * status_penalty
    """

    def __init__(
        self,
        encoder: BaseTextEncoder,
        alpha: float = 0.55,
        beta: float = 0.35,
        gamma: float = 0.30,
        top_k: int = 3,
    ):
        self.encoder = encoder
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.top_k = top_k

    def retrieve(self, memory: TemporalMemory) -> List[RetrievalResult]:
        ex = memory.example

        query_time = parse_time(ex.current_time)
        query_mode = infer_query_mode(ex.query, ex.task_type)

        query_vec = self.encoder.encode([ex.query])
        memory_texts = [item.text for item in memory.items]
        memory_vecs = self.encoder.encode(memory_texts)

        sims = cosine_np(query_vec, memory_vecs)

        results = []

        for item, sim in zip(memory.items, sims):
            t_score = temporal_relevance_score(item, query_time, query_mode)
            penalty = status_penalty(item, query_mode)

            score = (
                self.alpha * float(sim)
                + self.beta * float(t_score)
                - self.gamma * float(penalty)
            )

            results.append(
                RetrievalResult(
                    turn=item.turn,
                    text=item.text,
                    score=score,
                    semantic_score=float(sim),
                    temporal_score=float(t_score),
                    status_penalty=float(penalty),
                    validity_label=validity_label(item, query_time),
                    status=item.status,
                )
            )

        results.sort(key=lambda x: x.score, reverse=True)
        return results[: self.top_k]


# ============================================================
# Rule-based temporal answerer
# ============================================================

class RuleBasedTemporalAnswerer:
    """
    Deterministic diagnostic answerer.
    This is not the final LLM-agent method.
    It is useful for sanity checking dataset labels and retrieval.
    """

    def answer(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        task_type = example.task_type

        if task_type == "elapsed_time_reasoning":
            return self._answer_elapsed(example, memory, retrieved)

        if task_type == "aging_fact":
            return self._answer_aging(example, memory, retrieved)

        if task_type == "delayed_observation":
            return self._answer_delayed(example, memory, retrieved)

        if task_type == "rescheduling":
            return self._answer_rescheduling(example, memory, retrieved)

        if task_type == "cancellation":
            return self._answer_cancellation(example, memory, retrieved)

        if task_type == "time_window_retrieval":
            return self._answer_time_window(example, memory, retrieved)

        if task_type == "periodic_event":
            return self._answer_periodic(example, memory, retrieved)

        if task_type == "conflicting_updates":
            return self._answer_conflicting(example, memory, retrieved)

        return self._fallback_answer(example, memory, retrieved)

    def _get_item_by_turn(self, memory: TemporalMemory, turn: int) -> Optional[MemoryItem]:
        for item in memory.items:
            if item.turn == turn:
                return item
        return None

    def _retrieved_items(
        self,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> List[MemoryItem]:
        items = []

        for r in retrieved:
            item = self._get_item_by_turn(memory, r.turn)
            if item is not None:
                items.append(item)

        return items

    def _latest_active_item(self, memory: TemporalMemory) -> Optional[MemoryItem]:
        candidates = []

        for item in memory.items:
            status = normalize_text(item.status)

            if status in ACTIVE_STATUSES:
                key_time = item.valid_from or item.observation_time or item.event_time
                candidates.append((key_time, item))

        if not candidates:
            return None

        candidates.sort(
            key=lambda x: x[0] if x[0] is not None else parse_time("1900-01-01")
        )
        return candidates[-1][1]

    def _latest_item_by_observation(self, memory: TemporalMemory) -> Optional[MemoryItem]:
        candidates = []

        for item in memory.items:
            key_time = item.observation_time or item.valid_from or item.event_time
            candidates.append((key_time, item))

        if not candidates:
            return None

        candidates.sort(
            key=lambda x: x[0] if x[0] is not None else parse_time("1900-01-01")
        )
        return candidates[-1][1]

    def _answer_elapsed(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        query_time = parse_time(example.current_time)

        items = self._retrieved_items(memory, retrieved)
        if not items:
            items = memory.items

        for item in items:
            if item.valid_to is not None and query_time is not None:
                return "yes" if query_time >= item.valid_to else "no"

        return self._fallback_answer(example, memory, retrieved)

    def _answer_aging(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        query_time = parse_time(example.current_time)

        items = self._retrieved_items(memory, retrieved)
        if not items:
            items = memory.items

        for item in items:
            if item.valid_to is not None and query_time is not None:
                return "yes" if query_time <= item.valid_to else "no"

            if normalize_text(item.status) == "expired":
                return "no"

            if normalize_text(item.status) == "active":
                return "yes"

        return self._fallback_answer(example, memory, retrieved)

    def _answer_delayed(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        items = self._retrieved_items(memory, retrieved)
        if not items:
            items = memory.items

        for item in items:
            if item.event_time is not None:
                return item.event_time.strftime("%Y-%m-%dT%H:%M:%S")

        return self._fallback_answer(example, memory, retrieved)

    def _answer_rescheduling(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        item = self._latest_active_item(memory)

        if item is not None and item.event_time is not None:
            return item.event_time.strftime("%Y-%m-%dT%H:%M:%S")

        return self._fallback_answer(example, memory, retrieved)

    def _answer_cancellation(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        latest = self._latest_item_by_observation(memory)

        if latest is not None and normalize_text(latest.status) == "cancelled":
            return "no"

        active = self._latest_active_item(memory)

        if active is not None:
            return "yes"

        return "no"

    def _answer_time_window(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        query_time = self._extract_time_from_query(example.query)
        if query_time is None:
            query_time = parse_time(example.current_time)

        valid_items = []

        for item in memory.items:
            label = validity_label(item, query_time)
            if label == "valid":
                valid_items.append(item)

        if valid_items:
            valid_items.sort(
                key=lambda x: x.valid_from
                if x.valid_from is not None
                else parse_time("1900-01-01")
            )
            selected = valid_items[-1]
            return self._extract_content_answer(selected.text)

        return self._fallback_answer(example, memory, retrieved)

    def _answer_periodic(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        query_time = parse_time(example.current_time)

        items = self._retrieved_items(memory, retrieved)
        if not items:
            items = memory.items

        for item in items:
            if item.valid_to is not None and query_time is not None:
                return "yes" if query_time <= item.valid_to else "no"

            if normalize_text(item.status) == "expired":
                return "no"

            if normalize_text(item.status) == "active":
                return "yes"

        return self._fallback_answer(example, memory, retrieved)

    def _answer_conflicting(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        item = self._latest_active_item(memory)

        if item is not None:
            version = self._extract_version(item.text)
            if version is not None:
                return version

            return self._extract_content_answer(item.text)

        return self._fallback_answer(example, memory, retrieved)

    def _fallback_answer(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
    ) -> str:
        if retrieved:
            top_turn = retrieved[0].turn
            item = self._get_item_by_turn(memory, top_turn)
            if item is not None:
                if item.event_time is not None:
                    return item.event_time.strftime("%Y-%m-%dT%H:%M:%S")
                return self._extract_content_answer(item.text)

        return ""

    def _extract_time_from_query(self, query: str) -> Optional[Any]:
        candidates = re.findall(
            r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}",
            query,
        )

        if candidates:
            return parse_time(candidates[0])

        return None

    def _extract_version(self, text: str) -> Optional[str]:
        m = re.search(r"version\s+([0-9]+(?:\.[0-9]+)*)", text, flags=re.I)
        if m:
            return f"version {m.group(1)}"
        return None

    def _extract_content_answer(self, text: str) -> str:
        text = str(text)

        m = re.search(r"code\s+([A-Za-z0-9\-]+)", text, flags=re.I)
        if m:
            return m.group(1)

        m = re.search(r"(Office\s+[A-Z])", text)
        if m:
            return m.group(1)

        version = self._extract_version(text)
        if version:
            return version

        m = re.search(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}", text)
        if m:
            return m.group(0)

        if ":" in text:
            return text.split(":")[-1].strip()

        return text.strip()


# ============================================================
# Temporal consistency controller
# ============================================================

class TemporalConsistencyController:
    def check_contradiction(
        self,
        example: TimeBoundExample,
        memory: TemporalMemory,
        retrieved: List[RetrievalResult],
        predicted_answer: str,
    ) -> bool:
        task_type = example.task_type
        query_mode = infer_query_mode(example.query, task_type)
        query_time = parse_time(example.current_time)

        if not retrieved:
            return True

        top = retrieved[0]
        top_item = None

        for item in memory.items:
            if item.turn == top.turn:
                top_item = item
                break

        if top_item is None:
            return True

        status = normalize_text(top_item.status)
        label = validity_label(top_item, query_time)

        if query_mode in {"current", "prospective"}:
            if status in {"superseded", "expired"}:
                return True

            if label == "expired" and status != "cancelled":
                return True

        if task_type == "cancellation":
            latest = self._latest_by_observation(memory)
            if latest is not None:
                if normalize_text(latest.status) == "cancelled":
                    if normalize_text(predicted_answer) not in {"no", "cancelled"}:
                        return True

        return False

    def _latest_by_observation(self, memory: TemporalMemory) -> Optional[MemoryItem]:
        if not memory.items:
            return None

        return sorted(
            memory.items,
            key=lambda x: x.observation_time
            if x.observation_time is not None
            else parse_time("1900-01-01"),
        )[-1]


# ============================================================
# Metrics
# ============================================================

def answer_match(pred: str, gold: str) -> bool:
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)

    if pred_n == gold_n:
        return True

    if soft_numeric_match(pred, gold):
        return True

    if pred_n in gold_n or gold_n in pred_n:
        if len(pred_n) >= 2 and len(gold_n) >= 2:
            return True

    return False


def evidence_scores(
    predicted_turns: List[int],
    gold_turns: List[int],
) -> Tuple[float, float, float]:
    pred = set(predicted_turns)
    gold = set(gold_turns)

    if not pred and not gold:
        return 1.0, 1.0, 1.0

    if not pred:
        return 0.0, 0.0, 0.0

    if not gold:
        return 0.0, 0.0, 0.0

    tp = len(pred & gold)

    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold) if gold else 0.0

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


def aggregate_metrics(predictions: List[Prediction]) -> Dict[str, Any]:
    n = len(predictions)

    if n == 0:
        return {}

    acc = np.mean([p.answer_correct for p in predictions])
    ev_p = np.mean([p.evidence_precision for p in predictions])
    ev_r = np.mean([p.evidence_recall for p in predictions])
    ev_f1 = np.mean([p.evidence_f1 for p in predictions])
    contradiction_rate = np.mean([p.contradiction for p in predictions])

    return {
        "n_examples": n,
        "answer_accuracy": float(acc),
        "evidence_precision": float(ev_p),
        "evidence_recall": float(ev_r),
        "evidence_f1": float(ev_f1),
        "contradiction_rate": float(contradiction_rate),
    }


def metrics_by_group(
    predictions: List[Prediction],
    group_name: str,
) -> pd.DataFrame:
    groups = defaultdict(list)

    for p in predictions:
        key = getattr(p, group_name)
        groups[key].append(p)

    rows = []

    for key, ps in groups.items():
        m = aggregate_metrics(ps)
        m[group_name] = key
        rows.append(m)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    cols = [group_name] + [c for c in df.columns if c != group_name]
    return df[cols].sort_values(group_name)


# ============================================================
# Full TimeBound diagnostic method runner
# ============================================================

class TimeBoundMethod:
    def __init__(
        self,
        encoder: BaseTextEncoder,
        alpha: float = 0.55,
        beta: float = 0.35,
        gamma: float = 0.30,
        top_k: int = 3,
    ):
        self.retriever = TimeAwareRetriever(
            encoder=encoder,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
            top_k=top_k,
        )
        self.answerer = RuleBasedTemporalAnswerer()
        self.controller = TemporalConsistencyController()

    def run_one(self, example: TimeBoundExample) -> Prediction:
        memory = TemporalMemory(example)

        retrieved = self.retriever.retrieve(memory)
        predicted_answer = self.answerer.answer(example, memory, retrieved)

        predicted_turns = [r.turn for r in retrieved]
        precision, recall, f1 = evidence_scores(
            predicted_turns=predicted_turns,
            gold_turns=example.gold_evidence_turns,
        )

        correct = answer_match(predicted_answer, example.gold_answer)

        contradiction = self.controller.check_contradiction(
            example=example,
            memory=memory,
            retrieved=retrieved,
            predicted_answer=predicted_answer,
        )

        return Prediction(
            id=example.id,
            task_type=example.task_type,
            difficulty=example.difficulty,
            query=example.query,
            gold_answer=example.gold_answer,
            predicted_answer=predicted_answer,
            gold_evidence_turns=example.gold_evidence_turns,
            predicted_evidence_turns=predicted_turns,
            answer_correct=bool(correct),
            evidence_precision=float(precision),
            evidence_recall=float(recall),
            evidence_f1=float(f1),
            contradiction=bool(contradiction),
            retrieved=[asdict(r) for r in retrieved],
        )

    def run(self, examples: List[TimeBoundExample]) -> List[Prediction]:
        predictions = []

        for ex in tqdm(examples, desc="Evaluating TimeBound diagnostic baseline"):
            try:
                predictions.append(self.run_one(ex))
            except Exception as e:
                print(f"Failed example {ex.id}: {e}")

        return predictions


# ============================================================
# Saving
# ============================================================

def save_predictions(predictions: List[Prediction], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for p in predictions:
            f.write(json.dumps(asdict(p), ensure_ascii=False) + "\n")


def save_json(obj: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def print_metrics(metrics: Dict[str, Any]) -> None:
    print("\n" + "=" * 80)
    print("METRICS")
    print("=" * 80)

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    print("=" * 80)


# ============================================================
# CLI
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input",
        type=str,
        default=r"D:\Users\TimeBound\synthetic\timebound_synthetic_openai.jsonl",
        help="Path to TimeBound-Synthetic JSONL file.",
    )

    parser.add_argument(
        "--output_dir",
        type=str,
        default="timebound_method_outputs_no_sklearn",
        help="Directory for predictions and metrics.",
    )

    parser.add_argument(
        "--encoder",
        type=str,
        default="simple-tfidf",
        choices=["auto", "simple-tfidf", "sentence-transformers"],
        help="Text encoder for semantic retrieval.",
    )

    parser.add_argument(
        "--sentence_model",
        type=str,
        default="sentence-transformers/all-MiniLM-L6-v2",
        help="SentenceTransformer model name.",
    )

    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="Optional limit for quick debugging.",
    )

    parser.add_argument(
        "--top_k",
        type=int,
        default=3,
        help="Number of retrieved memory items.",
    )

    parser.add_argument(
        "--alpha",
        type=float,
        default=0.55,
        help="Semantic similarity weight.",
    )

    parser.add_argument(
        "--beta",
        type=float,
        default=0.35,
        help="Temporal relevance weight.",
    )

    parser.add_argument(
        "--gamma",
        type=float,
        default=0.30,
        help="Status penalty weight.",
    )

    args, unknown = parser.parse_known_args()

    input_path = Path(args.input)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("Loading examples")
    print("=" * 80)
    print(f"Input: {input_path}")

    examples = load_examples(input_path, limit=args.limit)
    print(f"Loaded examples: {len(examples)}")

    if len(examples) == 0:
        print("No examples loaded.")
        return

    print("\nTask distribution:")
    for k, v in Counter([ex.task_type for ex in examples]).most_common():
        print(f"  {k}: {v}")

    print("\nDifficulty distribution:")
    for k, v in Counter([ex.difficulty for ex in examples]).most_common():
        print(f"  {k}: {v}")

    print("\n" + "=" * 80)
    print("Building encoder")
    print("=" * 80)

    encoder = build_encoder(
        examples=examples,
        encoder_type=args.encoder,
        sentence_model_name=args.sentence_model,
    )

    print("\n" + "=" * 80)
    print("Running TimeBound diagnostic baseline")
    print("=" * 80)

    method = TimeBoundMethod(
        encoder=encoder,
        alpha=args.alpha,
        beta=args.beta,
        gamma=args.gamma,
        top_k=args.top_k,
    )

    predictions = method.run(examples)

    metrics = aggregate_metrics(predictions)
    print_metrics(metrics)

    by_task = metrics_by_group(predictions, "task_type")
    by_diff = metrics_by_group(predictions, "difficulty")

    pred_path = output_dir / "predictions.jsonl"
    metrics_path = output_dir / "metrics.json"
    by_task_path = output_dir / "metrics_by_task.csv"
    by_diff_path = output_dir / "metrics_by_difficulty.csv"

    save_predictions(predictions, pred_path)
    save_json(metrics, metrics_path)

    by_task.to_csv(by_task_path, index=False, encoding="utf-8")
    by_diff.to_csv(by_diff_path, index=False, encoding="utf-8")

    print("\nSaved:")
    print(f"  predictions:        {pred_path}")
    print(f"  metrics:            {metrics_path}")
    print(f"  metrics by task:    {by_task_path}")
    print(f"  metrics difficulty: {by_diff_path}")

    print("\nMetrics by task:")
    print(by_task.to_string(index=False))

    print("\nMetrics by difficulty:")
    print(by_diff.to_string(index=False))


if __name__ == "__main__":
    main()

Loading examples
Input: D:\Users\TimeBound\synthetic\timebound_synthetic_openai.jsonl
Loaded examples: 390

Task distribution:
  time_window_retrieval: 59
  elapsed_time_reasoning: 52
  periodic_event: 52
  conflicting_updates: 48
  delayed_observation: 47
  aging_fact: 46
  cancellation: 44
  rescheduling: 42

Difficulty distribution:
  easy: 138
  medium: 130
  hard: 122

Building encoder
Using SimpleTfidfTextEncoder without sklearn.

Running TimeBound diagnostic baseline


Evaluating TimeBound diagnostic baseline: 100%|█████████████████████████████████████| 390/390 [00:00<00:00, 794.30it/s]


Failed example timebound_openai_000319: can't compare offset-naive and offset-aware datetimes

METRICS
n_examples: 389
answer_accuracy: 0.4036
evidence_precision: 0.7721
evidence_recall: 0.9964
evidence_f1: 0.8453
contradiction_rate: 0.0540

Saved:
  predictions:        timebound_method_outputs_no_sklearn\predictions.jsonl
  metrics:            timebound_method_outputs_no_sklearn\metrics.json
  metrics by task:    timebound_method_outputs_no_sklearn\metrics_by_task.csv
  metrics difficulty: timebound_method_outputs_no_sklearn\metrics_by_difficulty.csv

Metrics by task:
             task_type  n_examples  answer_accuracy  evidence_precision  evidence_recall  evidence_f1  contradiction_rate
            aging_fact          46         0.456522            0.724638         1.000000     0.817391            0.239130
          cancellation          44         0.840909            0.784091         0.986742     0.850541            0.045455
   conflicting_updates          48         0.583333       

# Шаг 2. Сделать ablation retrieval

In [6]:
import json
import time
from pathlib import Path
from dataclasses import asdict
from typing import Dict, Any, List

import pandas as pd

# ВАЖНО:
# файл timebound_method_full_no_sklearn.py должен лежать рядом
# с этим ablation-скриптом
import timebound_method_full_no_sklearn as tb


# ============================================================
# Config
# ============================================================

INPUT_PATH = Path(r"D:\Users\TimeBound\synthetic\timebound_synthetic_openai.jsonl")

OUTPUT_ROOT = Path("timebound_retrieval_ablation_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ENCODER_TYPE = "simple-tfidf"
SENTENCE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 3

# Можно поставить None для полного прогона
LIMIT = None

ABLATIONS = {
    "semantic_only": {
        "alpha": 1.0,
        "beta": 0.0,
        "gamma": 0.0,
        "description": "Semantic retrieval only; ignores temporal validity and status.",
    },
    "temporal_only": {
        "alpha": 0.0,
        "beta": 1.0,
        "gamma": 0.0,
        "description": "Temporal relevance only; ignores semantic similarity and status penalty.",
    },
    "temporal_status": {
        "alpha": 0.0,
        "beta": 1.0,
        "gamma": 0.30,
        "description": "Temporal relevance with penalty for expired, superseded, or cancelled facts.",
    },
    "semantic_temporal": {
        "alpha": 0.60,
        "beta": 0.40,
        "gamma": 0.0,
        "description": "Semantic plus temporal retrieval without status penalty.",
    },
    "full_timebound": {
        "alpha": 0.55,
        "beta": 0.35,
        "gamma": 0.30,
        "description": "Full TimeBound retrieval: semantic similarity + temporal relevance - status penalty.",
    },
}


# ============================================================
# Helpers
# ============================================================

def save_predictions(predictions: List[tb.Prediction], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for p in predictions:
            f.write(json.dumps(asdict(p), ensure_ascii=False) + "\n")


def save_json(obj: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def invalid_evidence_rate(predictions: List[tb.Prediction]) -> float:
    """
    Counts how often top retrieved item is temporally problematic.

    This is not the same as contradiction_rate.
    It directly checks whether the top-1 retrieved item is expired/superseded
    or has an expired validity label.
    """
    if not predictions:
        return 0.0

    bad = 0

    for p in predictions:
        if not p.retrieved:
            bad += 1
            continue

        top = p.retrieved[0]

        status = str(top.get("status", "")).lower()
        validity = str(top.get("validity_label", "")).lower()

        if status in {"expired", "superseded"}:
            bad += 1
        elif validity == "expired" and status != "cancelled":
            bad += 1

    return bad / len(predictions)


def top1_evidence_hit_rate(predictions: List[tb.Prediction]) -> float:
    """
    Whether top-1 retrieved turn is among gold evidence turns.
    """
    if not predictions:
        return 0.0

    hits = 0

    for p in predictions:
        if not p.predicted_evidence_turns:
            continue

        top1 = p.predicted_evidence_turns[0]

        if top1 in set(p.gold_evidence_turns):
            hits += 1

    return hits / len(predictions)


def oracle_evidence_in_topk_rate(predictions: List[tb.Prediction]) -> float:
    """
    Whether at least one gold evidence turn appears in retrieved top-k.
    """
    if not predictions:
        return 0.0

    hits = 0

    for p in predictions:
        pred = set(p.predicted_evidence_turns)
        gold = set(p.gold_evidence_turns)

        if pred & gold:
            hits += 1

    return hits / len(predictions)


def add_extra_retrieval_metrics(metrics: Dict[str, Any], predictions: List[tb.Prediction]) -> Dict[str, Any]:
    metrics = dict(metrics)
    metrics["top1_evidence_hit_rate"] = top1_evidence_hit_rate(predictions)
    metrics["gold_in_topk_rate"] = oracle_evidence_in_topk_rate(predictions)
    metrics["invalid_evidence_rate"] = invalid_evidence_rate(predictions)
    return metrics


def run_single_ablation(
    name: str,
    params: Dict[str, Any],
    examples: List[tb.TimeBoundExample],
    encoder: tb.BaseTextEncoder,
) -> Dict[str, Any]:
    print("\n" + "=" * 100)
    print(f"Running ablation: {name}")
    print("=" * 100)
    print(params["description"])
    print(f"alpha={params['alpha']} beta={params['beta']} gamma={params['gamma']} top_k={TOP_K}")

    out_dir = OUTPUT_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)

    method = tb.TimeBoundMethod(
        encoder=encoder,
        alpha=params["alpha"],
        beta=params["beta"],
        gamma=params["gamma"],
        top_k=TOP_K,
    )

    t0 = time.time()
    predictions = method.run(examples)
    runtime_s = time.time() - t0

    metrics = tb.aggregate_metrics(predictions)
    metrics = add_extra_retrieval_metrics(metrics, predictions)

    metrics["ablation"] = name
    metrics["description"] = params["description"]
    metrics["alpha"] = params["alpha"]
    metrics["beta"] = params["beta"]
    metrics["gamma"] = params["gamma"]
    metrics["top_k"] = TOP_K
    metrics["runtime_s"] = runtime_s

    by_task = tb.metrics_by_group(predictions, "task_type")
    by_diff = tb.metrics_by_group(predictions, "difficulty")

    # Добавляем имя ablation в group tables
    if not by_task.empty:
        by_task.insert(0, "ablation", name)

    if not by_diff.empty:
        by_diff.insert(0, "ablation", name)

    save_predictions(predictions, out_dir / "predictions.jsonl")
    save_json(metrics, out_dir / "metrics.json")

    by_task.to_csv(out_dir / "metrics_by_task.csv", index=False, encoding="utf-8")
    by_diff.to_csv(out_dir / "metrics_by_difficulty.csv", index=False, encoding="utf-8")

    print("\nMetrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

    print(f"\nSaved to: {out_dir}")

    return {
        "metrics": metrics,
        "by_task": by_task,
        "by_difficulty": by_diff,
    }


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 100)
    print("TimeBound Retrieval Ablation")
    print("=" * 100)

    print(f"Input: {INPUT_PATH}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Limit: {LIMIT}")
    print(f"Top-k: {TOP_K}")

    examples = tb.load_examples(INPUT_PATH, limit=LIMIT)

    if not examples:
        raise RuntimeError(f"No examples loaded from {INPUT_PATH}")

    print(f"\nLoaded examples: {len(examples)}")

    print("\nTask distribution:")
    task_counts = {}
    for ex in examples:
        task_counts[ex.task_type] = task_counts.get(ex.task_type, 0) + 1

    for k, v in sorted(task_counts.items()):
        print(f"  {k}: {v}")

    print("\nBuilding shared encoder...")
    encoder = tb.build_encoder(
        examples=examples,
        encoder_type=ENCODER_TYPE,
        sentence_model_name=SENTENCE_MODEL,
    )

    all_metrics = []
    all_by_task = []
    all_by_diff = []

    for name, params in ABLATIONS.items():
        result = run_single_ablation(
            name=name,
            params=params,
            examples=examples,
            encoder=encoder,
        )

        all_metrics.append(result["metrics"])

        if not result["by_task"].empty:
            all_by_task.append(result["by_task"])

        if not result["by_difficulty"].empty:
            all_by_diff.append(result["by_difficulty"])

    summary_df = pd.DataFrame(all_metrics)

    # Удобный порядок колонок
    preferred_cols = [
        "ablation",
        "n_examples",
        "answer_accuracy",
        "evidence_precision",
        "evidence_recall",
        "evidence_f1",
        "top1_evidence_hit_rate",
        "gold_in_topk_rate",
        "invalid_evidence_rate",
        "contradiction_rate",
        "alpha",
        "beta",
        "gamma",
        "top_k",
        "runtime_s",
        "description",
    ]

    cols = [c for c in preferred_cols if c in summary_df.columns]
    cols += [c for c in summary_df.columns if c not in cols]
    summary_df = summary_df[cols]

    summary_path = OUTPUT_ROOT / "ablation_summary.csv"
    summary_json_path = OUTPUT_ROOT / "ablation_summary.json"

    summary_df.to_csv(summary_path, index=False, encoding="utf-8")

    with summary_json_path.open("w", encoding="utf-8") as f:
        json.dump(all_metrics, f, ensure_ascii=False, indent=2)

    if all_by_task:
        by_task_all = pd.concat(all_by_task, ignore_index=True)
        by_task_all.to_csv(OUTPUT_ROOT / "ablation_by_task.csv", index=False, encoding="utf-8")
    else:
        by_task_all = pd.DataFrame()

    if all_by_diff:
        by_diff_all = pd.concat(all_by_diff, ignore_index=True)
        by_diff_all.to_csv(OUTPUT_ROOT / "ablation_by_difficulty.csv", index=False, encoding="utf-8")
    else:
        by_diff_all = pd.DataFrame()

    print("\n" + "=" * 100)
    print("ABLATION SUMMARY")
    print("=" * 100)
    print(summary_df.to_string(index=False))

    print("\nSaved:")
    print(f"  {summary_path}")
    print(f"  {summary_json_path}")
    print(f"  {OUTPUT_ROOT / 'ablation_by_task.csv'}")
    print(f"  {OUTPUT_ROOT / 'ablation_by_difficulty.csv'}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TimeBound Retrieval Ablation
Input: D:\Users\TimeBound\synthetic\timebound_synthetic_openai.jsonl
Output root: timebound_retrieval_ablation_outputs
Limit: None
Top-k: 3

Loaded examples: 390

Task distribution:
  aging_fact: 46
  cancellation: 44
  conflicting_updates: 48
  delayed_observation: 47
  elapsed_time_reasoning: 52
  periodic_event: 52
  rescheduling: 42
  time_window_retrieval: 59

Building shared encoder...
Using SimpleTfidfTextEncoder without sklearn.

Running ablation: semantic_only
Semantic retrieval only; ignores temporal validity and status.
alpha=1.0 beta=0.0 gamma=0.0 top_k=3


Evaluating TimeBound diagnostic baseline: 100%|████████████████████████████████████| 390/390 [00:00<00:00, 1104.82it/s]


Failed example timebound_openai_000319: can't compare offset-naive and offset-aware datetimes

Metrics:
  n_examples: 389
  answer_accuracy: 0.4422
  evidence_precision: 0.7712
  evidence_recall: 0.9951
  evidence_f1: 0.8442
  contradiction_rate: 0.0977
  top1_evidence_hit_rate: 0.7635
  gold_in_topk_rate: 1.0000
  invalid_evidence_rate: 0.2905
  ablation: semantic_only
  description: Semantic retrieval only; ignores temporal validity and status.
  alpha: 1.0000
  beta: 0.0000
  gamma: 0.0000
  top_k: 3
  runtime_s: 0.3580

Saved to: timebound_retrieval_ablation_outputs\semantic_only

Running ablation: temporal_only
Temporal relevance only; ignores semantic similarity and status penalty.
alpha=0.0 beta=1.0 gamma=0.0 top_k=3


Evaluating TimeBound diagnostic baseline: 100%|████████████████████████████████████| 390/390 [00:00<00:00, 1214.96it/s]


Failed example timebound_openai_000319: can't compare offset-naive and offset-aware datetimes

Metrics:
  n_examples: 389
  answer_accuracy: 0.4062
  evidence_precision: 0.7686
  evidence_recall: 0.9904
  evidence_f1: 0.8411
  contradiction_rate: 0.0488
  top1_evidence_hit_rate: 0.7943
  gold_in_topk_rate: 0.9974
  invalid_evidence_rate: 0.2005
  ablation: temporal_only
  description: Temporal relevance only; ignores semantic similarity and status penalty.
  alpha: 0.0000
  beta: 1.0000
  gamma: 0.0000
  top_k: 3
  runtime_s: 0.3230

Saved to: timebound_retrieval_ablation_outputs\temporal_only

Running ablation: temporal_status
Temporal relevance with penalty for expired, superseded, or cancelled facts.
alpha=0.0 beta=1.0 gamma=0.3 top_k=3


...

Failed example timebound_openai_000319: can't compare offset-naive and offset-aware datetimes

Metrics:
  n_examples: 389
  answer_accuracy: 0.4062
  evidence_precision: 0.7686
  evidence_recall: 0.9904
  evidence_f1: 0.8411
  contradiction_rate: 0.0488
  top1_evidence_hit_rate: 0.7892
  gold_in_topk_rate: 0.9974
  invalid_evidence_rate: 0.2005
  ablation: temporal_status
  description: Temporal relevance with penalty for expired, superseded, or cancelled facts.
  alpha: 0.0000
  beta: 1.0000
  gamma: 0.3000
  top_k: 3
  runtime_s: 0.3570

Saved to: timebound_retrieval_ablation_outputs\temporal_status

Running ablation: semantic_temporal
Semantic plus temporal retrieval without status penalty.
alpha=0.6 beta=0.4 gamma=0.0 top_k=3


Evaluating TimeBound diagnostic baseline: 100%|████████████████████████████████████| 390/390 [00:00<00:00, 1143.70it/s]


Failed example timebound_openai_000319: can't compare offset-naive and offset-aware datetimes

Metrics:
  n_examples: 389
  answer_accuracy: 0.4036
  evidence_precision: 0.7721
  evidence_recall: 0.9964
  evidence_f1: 0.8453
  contradiction_rate: 0.0488
  top1_evidence_hit_rate: 0.7815
  gold_in_topk_rate: 1.0000
  invalid_evidence_rate: 0.1954
  ablation: semantic_temporal
  description: Semantic plus temporal retrieval without status penalty.
  alpha: 0.6000
  beta: 0.4000
  gamma: 0.0000
  top_k: 3
  runtime_s: 0.3470

Saved to: timebound_retrieval_ablation_outputs\semantic_temporal

Running ablation: full_timebound
Full TimeBound retrieval: semantic similarity + temporal relevance - status penalty.
alpha=0.55 beta=0.35 gamma=0.3 top_k=3


Evaluating TimeBound diagnostic baseline: 100%|████████████████████████████████████| 390/390 [00:00<00:00, 1238.09it/s]


Failed example timebound_openai_000319: can't compare offset-naive and offset-aware datetimes

Metrics:
  n_examples: 389
  answer_accuracy: 0.4036
  evidence_precision: 0.7721
  evidence_recall: 0.9964
  evidence_f1: 0.8453
  contradiction_rate: 0.0540
  top1_evidence_hit_rate: 0.7635
  gold_in_topk_rate: 1.0000
  invalid_evidence_rate: 0.2108
  ablation: full_timebound
  description: Full TimeBound retrieval: semantic similarity + temporal relevance - status penalty.
  alpha: 0.5500
  beta: 0.3500
  gamma: 0.3000
  top_k: 3
  runtime_s: 0.3170

Saved to: timebound_retrieval_ablation_outputs\full_timebound

ABLATION SUMMARY
         ablation  n_examples  answer_accuracy  evidence_precision  evidence_recall  evidence_f1  top1_evidence_hit_rate  gold_in_topk_rate  invalid_evidence_rate  contradiction_rate  alpha  beta  gamma  top_k  runtime_s                                                                          description
    semantic_only         389         0.442159            0.7

# Шаг 3. Посмотреть failures

In [4]:
import json
from pathlib import Path

path = Path("timebound_method_outputs_no_sklearn/predictions.jsonl")

bad = []

with path.open("r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        if not obj["answer_correct"]:
            bad.append(obj)

print("bad:", len(bad))

for x in bad[:10]:
    print("=" * 100)
    print("ID:", x["id"])
    print("TASK:", x["task_type"])
    print("QUERY:", x["query"])
    print("GOLD:", x["gold_answer"])
    print("PRED:", x["predicted_answer"])
    print("GOLD EVID:", x["gold_evidence_turns"])
    print("PRED EVID:", x["predicted_evidence_turns"])
    print("RETRIEVED:")
    for r in x["retrieved"]:
        print(" ", r["turn"], r["status"], r["validity_label"], round(r["score"], 3), r["text"])

bad: 232
ID: timebound_openai_000000
TASK: time_window_retrieval
QUERY: What appointments did I have scheduled on 2026-05-05?
GOLD: Doctor appointment at 14:00
PRED: 2026-05-05T14:00:00
GOLD EVID: [1]
PRED EVID: [1, 2]
RETRIEVED:
  1 scheduled expired 0.504 I have a doctor appointment on 2026-05-05 at 14:00.
  2 scheduled expired 0.309 Also, a project meeting is set for 2026-05-07 at 09:00.
ID: timebound_openai_000001
TASK: elapsed_time_reasoning
QUERY: How many days have I actively studied the course by 2026-06-15 10:00?
GOLD: 9 days
PRED: 2026-06-10T09:00:00
GOLD EVID: [1, 2]
PRED EVID: [2, 1]
RETRIEVED:
  2 active valid 0.526 I paused the course temporarily on 2026-06-10 at 09:00.
  1 active valid 0.48 I started a 2-week online course on 2026-06-01 at 08:00.
ID: timebound_openai_000002
TASK: delayed_observation
QUERY: Was the parcel delivered on 2027-01-15?
GOLD: No
PRED: 2027-01-14T20:00:00
GOLD EVID: [1]
PRED EVID: [1, 2]
RETRIEVED:
  1 delayed valid 0.626 I just learned that my p